# Classification Benchmark
Unified pipeline to train, evaluate, and compare multiple sklearn classifiers side-by-side.

**Metrics collected**
| Metric | Description |
|---|---|
| Accuracy | Fraction of correct predictions |
| Precision | Weighted avg positive predictive value |
| Recall | Weighted avg true positive rate |
| F1-Score | Harmonic mean of precision & recall |
| ROC-AUC | Area under the ROC curve (one-vs-rest, weighted) |
| Train time | Wall-clock fit duration (seconds) |
| Predict time | Wall-clock inference duration (seconds) |
| Peak memory | `tracemalloc` peak during training (KiB) |

In [1]:
import warnings, sys, os
sys.path.insert(0, "/home/claude/benchmarking")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                              precision_score, recall_score, roc_auc_score)
from sklearn.preprocessing import LabelBinarizer
from metrics_utils import (BenchmarkResult, highlight_best, memory_tracker,
                            prepare_split, print_summary_table,
                            results_to_dataframe, timer)
print("✓ Imports OK")

✓ Imports OK


## ClassificationBenchmark class

In [2]:
class ClassificationBenchmark:
    def __init__(self, X, y, dataset_name="Dataset", test_size=0.2, random_state=42):
        self.dataset_name = dataset_name
        self.X_train, self.X_test, self.y_train, self.y_test = prepare_split(
            np.asarray(X), np.asarray(y), test_size=test_size, random_state=random_state)
        self._n_classes = len(np.unique(y))
        self._models = []
        self.results = []

    def add_model(self, name, model):
        self._models.append((name, model))
        return self

    def run(self):
        self.results.clear()
        print(f"\n🔬  Benchmarking on '{self.dataset_name}'  "
              f"({self.X_train.shape[0]} train / {self.X_test.shape[0]} test samples)\n")
        for name, model in self._models:
            result = self._evaluate(name, model)
            self.results.append(result)
            print(f"  ✓  {name:<35}  acc={result.metrics['accuracy']:.4f}  "
                  f"f1={result.metrics['f1_score']:.4f}  "
                  f"train={result.train_time_s:.3f}s")
        return self

    def print_summary(self):
        df = results_to_dataframe(self.results)
        highlighted = highlight_best(
            df, ["accuracy","precision","recall","f1_score","roc_auc"],
               ["train_time_s","predict_time_s","peak_memory_kb"])
        print_summary_table(highlighted, title=f"Classification — {self.dataset_name}")
        return df

    def save_results(self, path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        df = results_to_dataframe(self.results)
        df.to_csv(path)
        print(f"  📄  Saved → {path}")
        return df

    def _evaluate(self, name, model):
        result = BenchmarkResult(model_name=name, task="classification")
        with memory_tracker() as mem, timer() as t:
            model.fit(self.X_train, self.y_train)
        result.train_time_s   = t["elapsed"]
        result.peak_memory_kb = mem["peak_kb"]
        with timer() as t:
            y_pred = model.predict(self.X_test)
        result.predict_time_s = t["elapsed"]
        avg = "binary" if self._n_classes == 2 else "weighted"
        result.metrics["accuracy"]  = accuracy_score(self.y_test, y_pred)
        result.metrics["precision"] = precision_score(self.y_test, y_pred, average=avg, zero_division=0)
        result.metrics["recall"]    = recall_score(self.y_test, y_pred, average=avg, zero_division=0)
        result.metrics["f1_score"]  = f1_score(self.y_test, y_pred, average=avg, zero_division=0)
        result.metrics["roc_auc"]   = self._roc_auc(model, y_pred)
        classes = np.unique(self.y_test)
        result.extra["confusion_matrix"] = confusion_matrix(self.y_test, y_pred, labels=classes)
        result.extra["classes"] = classes.tolist()
        return result

    def _roc_auc(self, model, y_pred):
        try:
            if hasattr(model, "predict_proba"):
                y_score = model.predict_proba(self.X_test)
                if self._n_classes == 2:
                    return roc_auc_score(self.y_test, y_score[:, 1])
                return roc_auc_score(self.y_test, y_score, multi_class="ovr", average="weighted")
            elif hasattr(model, "decision_function"):
                y_score = model.decision_function(self.X_test)
                if self._n_classes == 2:
                    return roc_auc_score(self.y_test, y_score)
                lb = LabelBinarizer()
                return roc_auc_score(lb.fit_transform(self.y_test), y_score, average="weighted")
        except Exception:
            pass
        try:
            return roc_auc_score(self.y_test, y_pred, average="weighted")
        except Exception:
            return float("nan")

print("✓ ClassificationBenchmark defined")

✓ ClassificationBenchmark defined


## Dataset 1 — Breast Cancer (binary)

In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

MODELS = [
    ("Logistic Regression",    LogisticRegression(max_iter=5000, random_state=42)),
    ("Decision Tree",          DecisionTreeClassifier(random_state=42)),
    ("Random Forest",          RandomForestClassifier(n_estimators=100, random_state=42)),
    ("Gradient Boosting",      GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ("K-Nearest Neighbours",   KNeighborsClassifier(n_neighbors=5)),
    ("Support Vector Machine", SVC(kernel="rbf", probability=True, random_state=42)),
    ("Gaussian Naive Bayes",   GaussianNB()),
]

X, y = load_breast_cancer(return_X_y=True)
bench_bc = ClassificationBenchmark(X, y, dataset_name="Breast Cancer")
for name, model in MODELS:
    bench_bc.add_model(name, model)
bench_bc.run()
df_bc = bench_bc.print_summary()
bench_bc.save_results("sample_results/classification_breast_cancer.csv")


🔬  Benchmarking on 'Breast Cancer'  (455 train / 114 test samples)



  ✓  Logistic Regression                  acc=0.9649  f1=0.9726  train=1.256s
  ✓  Decision Tree                        acc=0.9123  f1=0.9286  train=0.013s


  ✓  Random Forest                        acc=0.9561  f1=0.9655  train=1.153s


  ✓  Gradient Boosting                    acc=0.9561  f1=0.9660  train=0.665s
  ✓  K-Nearest Neighbours                 acc=0.9123  f1=0.9296  train=0.002s
  ✓  Support Vector Machine               acc=0.9298  f1=0.9459  train=0.027s
  ✓  Gaussian Naive Bayes                 acc=0.9386  f1=0.9517  train=0.004s

──────────────────────────────────────────────────────────────────────────────────────────
  Classification — Breast Cancer
──────────────────────────────────────────────────────────────────────────────────────────
                       train_time_s predict_time_s peak_memory_kb  accuracy precision    recall  f1_score   roc_auc
model                                                                                                              
Logistic Regression           1.256        0.00043           64.5  0.9649 ★  0.9595 ★  0.9861 ★  0.9726 ★  0.9954 ★
Decision Tree                0.0131       0.000374           86.1    0.9123    0.9559    0.9028    0.9286    0.9157
Random F

,train_time_s,predict_time_s,peak_memory_kb,accuracy,precision,recall,f1_score,roc_auc
model,,,,,,,,
Logistic Regression,1.2560,0.000430,64.5,0.9649,0.9595,0.9861,0.9726,0.9954
Decision Tree,0.0131,0.000374,86.1,0.9123,0.9559,0.9028,0.9286,0.9157
Random Forest,1.1534,0.009594,188.6,0.9561,0.9589,0.9722,0.9655,0.9937
Gradient Boosting,0.6654,0.000958,154.4,0.9561,0.9467,0.9861,0.9660,0.9907
K-Nearest Neighbours,0.0019,0.033319,30.0,0.9123,0.9429,0.9167,0.9296,0.9559
Support Vector Machine,0.0266,0.001616,117.3,0.9298,0.9211,0.9722,0.9459,0.9696
Gaussian Naive Bayes,0.0038,0.000344,206.8,0.9386,0.9452,0.9583,0.9517,0.9878


## Dataset 2 — Iris (multiclass)

In [4]:
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)
bench_iris = ClassificationBenchmark(X, y, dataset_name="Iris")
for name, model in [
    ("Logistic Regression",    LogisticRegression(max_iter=5000, random_state=42)),
    ("Decision Tree",          DecisionTreeClassifier(random_state=42)),
    ("Random Forest",          RandomForestClassifier(n_estimators=100, random_state=42)),
    ("Gradient Boosting",      GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ("K-Nearest Neighbours",   KNeighborsClassifier(n_neighbors=5)),
    ("Support Vector Machine", SVC(kernel="rbf", probability=True, random_state=42)),
    ("Gaussian Naive Bayes",   GaussianNB()),
]:
    bench_iris.add_model(name, model)
bench_iris.run()
df_iris = bench_iris.print_summary()
bench_iris.save_results("sample_results/classification_iris.csv")


🔬  Benchmarking on 'Iris'  (120 train / 30 test samples)



  ✓  Logistic Regression                  acc=0.9667  f1=0.9666  train=0.051s
  ✓  Decision Tree                        acc=0.9333  f1=0.9333  train=0.006s


  ✓  Random Forest                        acc=0.9000  f1=0.8997  train=1.159s


  ✓  Gradient Boosting                    acc=0.9667  f1=0.9666  train=0.943s
  ✓  K-Nearest Neighbours                 acc=1.0000  f1=1.0000  train=0.003s
  ✓  Support Vector Machine               acc=0.9667  f1=0.9666  train=0.008s
  ✓  Gaussian Naive Bayes                 acc=0.9667  f1=0.9666  train=0.003s

──────────────────────────────────────────────────────────────────────────────────────────
  Classification — Iris
──────────────────────────────────────────────────────────────────────────────────────────
                       train_time_s predict_time_s peak_memory_kb accuracy precision  recall f1_score roc_auc
model                                                                                                        
Logistic Regression          0.0508       0.001389           41.9   0.9667    0.9697  0.9667   0.9666   1.0 ★
Decision Tree                0.0064     0.000156 ★           20.5   0.9333    0.9333  0.9333   0.9333    0.95
Random Forest                 1.159      

,train_time_s,predict_time_s,peak_memory_kb,accuracy,precision,recall,f1_score,roc_auc
model,,,,,,,,
Logistic Regression,0.0508,0.001389,41.9,0.9667,0.9697,0.9667,0.9666,1.0000
Decision Tree,0.0064,0.000156,20.5,0.9333,0.9333,0.9333,0.9333,0.9500
Random Forest,1.1590,0.008037,119.3,0.9000,0.9024,0.9000,0.8997,0.9867
Gradient Boosting,0.9432,0.001704,172.6,0.9667,0.9697,0.9667,0.9666,0.9900
K-Nearest Neighbours,0.0033,0.001997,10.7,1.0000,1.0000,1.0000,1.0000,1.0000
Support Vector Machine,0.0076,0.000328,15.8,0.9667,0.9697,0.9667,0.9666,0.9967
Gaussian Naive Bayes,0.0035,0.000246,11.6,0.9667,0.9697,0.9667,0.9666,0.9900


## Visualizations

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "#f8f9fb",
    "axes.grid": True, "grid.color": "white", "grid.linewidth": 1.2,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "sans-serif",
})
PALETTE = sns.color_palette("muted")

def grouped_bar(df, metrics, title, path):
    models = df.index.tolist()
    x = np.arange(len(models))
    width = 0.8 / len(metrics)
    fig, ax = plt.subplots(figsize=(max(10, len(models)*1.4), 5), dpi=130)
    for i, metric in enumerate(metrics):
        vals = df[metric].values
        bars = ax.bar(x + (i - len(metrics)/2 + 0.5)*width, vals,
                      width*0.85, label=metric.replace("_"," ").title(),
                      color=PALETTE[i % len(PALETTE)], zorder=3)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=30, ha="right")
    ax.set_ylim(0, 1.1); ax.set_ylabel("Score"); ax.set_title(title)
    ax.legend(loc="lower right", ncol=len(metrics)); fig.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches="tight"); plt.show()
    print(f"  🖼  {path}")

def heatmap_plot(df, metrics, title, path):
    sub  = df[metrics].copy()
    norm = (sub - sub.min()) / (sub.max() - sub.min() + 1e-9)
    for col in ["train_time_s","predict_time_s","peak_memory_kb"]:
        if col in norm.columns: norm[col] = 1 - norm[col]
    fig, ax = plt.subplots(figsize=(len(metrics)*1.3+2, len(sub)*0.6+2), dpi=130)
    sns.heatmap(norm, annot=sub.round(4), fmt=".4f", cmap="YlOrRd_r",
                linewidths=0.4, linecolor="white", ax=ax,
                cbar_kws={"label":"Normalised rank (green=better)"},
                annot_kws={"size":9})
    ax.set_xticklabels([m.replace("_"," ").upper() for m in metrics], rotation=30, ha="right")
    ax.set_title(title, pad=12); fig.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches="tight"); plt.show()
    print(f"  🖼  {path}")

def confusion_grid(results, title, path):
    filt = [r for r in results if "confusion_matrix" in r.extra]
    ncols = min(len(filt), 4); nrows = (len(filt)+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*3.8), dpi=130)
    axes = np.array(axes).flatten()
    for ax, result in zip(axes, filt):
        cm = result.extra["confusion_matrix"]
        labels = result.extra.get("classes", list(range(cm.shape[0])))
        sns.heatmap(pd.DataFrame(cm, index=labels, columns=labels),
                    annot=True, fmt="d", cmap="Blues", ax=ax,
                    linewidths=0.5, cbar=False, annot_kws={"size":9})
        ax.set_title(result.model_name, fontsize=10)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    for ax in axes[len(filt):]: ax.set_visible(False)
    fig.suptitle(title, fontsize=13, y=1.01); fig.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches="tight"); plt.show()
    print(f"  🖼  {path}")

print("✓ Plot helpers defined")

✓ Plot helpers defined


In [6]:
# Breast Cancer — metric bar chart
METRICS_CLF = ["accuracy","precision","recall","f1_score","roc_auc"]
grouped_bar(df_bc, METRICS_CLF,
            "Classification metrics — Breast Cancer",
            "sample_results/clf_bc_metrics.png")

  🖼  sample_results/clf_bc_metrics.png


In [7]:
# Breast Cancer — heatmap
heatmap_plot(df_bc, METRICS_CLF,
             "Metric heatmap — Breast Cancer",
             "sample_results/clf_bc_heatmap.png")

  🖼  sample_results/clf_bc_heatmap.png


In [8]:
# Breast Cancer — confusion matrices
confusion_grid(bench_bc.results,
               "Confusion matrices — Breast Cancer",
               "sample_results/clf_bc_confusion.png")

  🖼  sample_results/clf_bc_confusion.png


In [9]:
# Iris — metric bar chart
grouped_bar(df_iris, METRICS_CLF,
            "Classification metrics — Iris",
            "sample_results/clf_iris_metrics.png")

  🖼  sample_results/clf_iris_metrics.png


In [10]:
# Iris — heatmap
heatmap_plot(df_iris, METRICS_CLF,
             "Metric heatmap — Iris",
             "sample_results/clf_iris_heatmap.png")

  🖼  sample_results/clf_iris_heatmap.png


In [11]:
# Speed vs Accuracy scatter (Breast Cancer)
fig, ax = plt.subplots(figsize=(8,5), dpi=130)
sizes = df_bc["f1_score"].values * 500 + 50
ax.scatter(df_bc["train_time_s"], df_bc["accuracy"], s=sizes,
           c=range(len(df_bc)), cmap="tab10", alpha=0.85,
           edgecolors="white", linewidths=0.8, zorder=3)
for model, row in df_bc.iterrows():
    ax.annotate(model, (row["train_time_s"], row["accuracy"]),
                textcoords="offset points", xytext=(8,4), fontsize=8)
ax.set_xlabel("Training time (s)"); ax.set_ylabel("Accuracy")
ax.set_title("Speed vs Accuracy — Breast Cancer")
ax.text(0.02, 0.02, "Marker size ∝ F1-score", transform=ax.transAxes, fontsize=8, color="#666")
fig.tight_layout()
fig.savefig("sample_results/clf_bc_speed.png", dpi=130, bbox_inches="tight")
plt.show(); print("  🖼  sample_results/clf_bc_speed.png")

  🖼  sample_results/clf_bc_speed.png


In [12]:
# Performance overhead bars (Breast Cancer)
overhead_cols = ["train_time_s","predict_time_s","peak_memory_kb"]
labels = {"train_time_s":"Train time (s)","predict_time_s":"Predict time (s)","peak_memory_kb":"Peak memory (KB)"}
fig, axes = plt.subplots(1, 3, figsize=(15, max(4, len(df_bc)*0.55+1.5)), dpi=130)
y = np.arange(len(df_bc))
for ax, col in zip(axes, overhead_cols):
    vals = df_bc[col].values
    bars = ax.barh(y, vals, color=PALETTE[overhead_cols.index(col)], alpha=0.85, zorder=3)
    ax.set_yticks(y); ax.set_yticklabels(df_bc.index); ax.set_xlabel(labels[col])
    ax.set_title(labels[col]); ax.invert_yaxis()
    for bar, val in zip(bars, vals):
        ax.text(val*1.01, bar.get_y()+bar.get_height()/2, f"{val:.4f}", va="center", fontsize=8)
fig.suptitle("Performance overhead — Breast Cancer", fontsize=13)
fig.tight_layout()
fig.savefig("sample_results/clf_bc_overhead.png", dpi=130, bbox_inches="tight")
plt.show(); print("  🖼  sample_results/clf_bc_overhead.png")

  🖼  sample_results/clf_bc_overhead.png
